# M2L_L3_EVAL — label-free per-recording protocol, SzScan 1.0 baseline (Track F, milestone M2, chat F2-03)

**Design:** `results/m2l/DESIGN_L2.md` (approved by the author 23/11/2026 14:19). This notebook: **test, chb24 and Siena, scored once** with the chosen arm, its k and the minimum length (results/m2l/l3short/minlen.json); V5 control in the same session.
Every quantity V5 fits per patient (z-score statistics, per-channel latent reference, percentiles, alarm budget) is
fitted on the recording itself; the four V5 checkpoints, inference only; raw EDF verified against PhysioNet in the
session; the V5 control (R1.0) and the code-path gates run first and STOP on any difference.
**Account:** `nhn2mm`. **Accelerator:** None (CPU). **Internet:** ON (pip, git clone, PhysioNet checksums).
**Inputs (Add Input):** `combined-siena-scalp-and-chb-mit-eeg-datasets`, `v9-x3-fold1`, `v9-x3-fold3`, `thesis-r2-external-features`.
**Output:** `/kaggle/working/m2l/l3eval` (→ `results/m2l/l3eval`; small files only). **Runtime:** ≈ 3–3.5 h (computed from L2's measured job times; PROJ stops above 10.5 h).
Code: `github.com/Nhanvas/Thesis` ref `main`; every module's sha256 is asserted below.
**Nothing printed here is a research result.** Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "M2L_L3_EVAL", "main"
TEST = bool(os.environ.get("EUR_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("EUR_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("EUR_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("EUR_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["EUR_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/m2l_run.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"m2l_core.py": "9690cac7f91f52b2", "m2l_run.py": "4263c38f81c7adb8", "eur_abl.py": "0fd303e9b2f3671f", "r2_r1.py": "76cbe51ff0180d4a", "r2_core.py": "902d15ad4feb3e8a", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "p3c_freeze.py": "dabe4139f6005fd2", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e22_test_oneshot.py": "669350bbfaaf0f4e", "e23_test_look7.py": "88587668baf12a64", "x1_external.py": "9962e3e7fe4ddc35", "p2_dump_seeds23.py": "0bc887542989310f", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "dataprep/preprocessing.py": "0172fbea61b28540", "continuous_eval.py": "8fea38dabee5ad25", "c1b_build_trainset.py": "b4596a8aa3797653", "evaluation_protocol.py": "7caacac9b31bd0f7"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", **{"M2L_MODE": "L3EVAL"})
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import m2l_run as M
assert M.__file__ == f"{REPO}/src/m2l_run.py", M.__file__
print("CPU cores:", os.cpu_count(), "| mode", os.environ["M2L_MODE"], "| m2l_core", M.M.sha16(M.M.__file__))

In [ ]:
# RUN: gates, smoke / control, per-recording build, scoring
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("M2L_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, projection, outputs, measured runtime
OUT = os.path.join(WORK, "m2l", "l3eval")
for f in ("gates.json", "projection.json", "k.json", "smoke.json", "run_meta.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")